# RiverScope / PlanetScope: exploration and quality control

This notebook inspects the public RiverScope dataset before using it for water segmentation and shoreline analysis. It preserves the original `0/1/2` labels and derives a binary mask only for visualization and statistics. It does not train models or modify the dataset.

It reviews bands and scales, CRS and nodata, image-mask alignment, visual boundary quality, artifacts, class balance, duplicates, and the geographic separation of splits.

In [ ]:
from pathlib import Path
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import BoundaryNorm, ListedColormap

from reservoir_segmentation.boundaries import (
    distance_squared_to_boundary,
    find_boundary_pixels,
    weights_from_distance_squared,
)
from reservoir_segmentation.dataset import (
    RIVERSCOPE_ARCHIVE_MD5,
    RIVERSCOPE_ARCHIVE_URL,
    RIVERSCOPE_RECORD_URL,
    download_riverscope_archive,
    extract_segmentation_archive,
    filter_riverscope_rows,
    find_riverscope_all_zero_images,
    load_quality_exclusions,
    materialize_riverscope_rows,
    materialize_riverscope_sample,
    read_riverscope_split,
)

SEED = 2026
SPLIT = "train"
SAMPLE_COUNT = 30
DOWNLOAD_DATASET = False  # True explicitly downloads approximately 8.1 GB.
def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml from the current directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_ROOT = Path(os.environ.get("RIVERSCOPE_DATA", PROJECT_ROOT / "data" / "raw" / "riverscope"))
ARCHIVE_PATH = PROJECT_ROOT / "data" / "raw" / "RiverScope.zip"
EXTRACTED_ROOT = PROJECT_ROOT / "data" / "raw" / "riverscope"
SAMPLE_ROOT = PROJECT_ROOT / "data" / "processed" / "riverscope_sample"
QUALITY_EXCLUSIONS_PATH = PROJECT_ROOT / "configs" / "riverscope_quality_exclusions.csv"
random.seed(SEED)
np.random.seed(SEED)
print("Official source:", RIVERSCOPE_RECORD_URL)

## Reproducible acquisition

The official download is the RiverScope Zenodo record. The ZIP contains the CSVs and co-registered products; this analysis uses the PlanetScope `normalized_planetscope_path` and `label_path` fields. Downloading does not run automatically.

For the most faithful copy of the published dataset, use the complete official ZIP and do not recompress or convert the GeoTIFFs before inspecting them. The function validates the published MD5.

In [ ]:
if DOWNLOAD_DATASET:
    download_riverscope_archive(ARCHIVE_PATH)
    extract_segmentation_archive(ARCHIVE_PATH, EXTRACTED_ROOT)
    DATA_SOURCE = EXTRACTED_ROOT
elif ARCHIVE_PATH.is_file():
    DATA_SOURCE = ARCHIVE_PATH
elif DATA_ROOT.is_dir():
    DATA_SOURCE = DATA_ROOT
else:
    DATA_SOURCE = None

if DATA_SOURCE is None:
    print("No local data found. Download:", RIVERSCOPE_ARCHIVE_URL)
    print("Expected MD5:", RIVERSCOPE_ARCHIVE_MD5)
    print("Place the ZIP in data/raw/RiverScope.zip or extract it into data/raw/riverscope.")
else:
    print("Local source:", DATA_SOURCE)

## Official CSVs and reproducible sample

The CSVs are the source of truth for pairing each tile with its label; files are not paired by filename. The sample retains `reach_id`, coordinates, `planetscope_id`, and the original paths. Only the selected pairs are extracted when the source is a ZIP. A versioned quality-exclusion manifest is applied only to derived samples and downstream workflows; the official CSVs remain unchanged.

In [ ]:
quality_exclusions = load_quality_exclusions(QUALITY_EXCLUSIONS_PATH)
if DATA_SOURCE is None:
    split_tables, eligible_split_tables, sample_rows = {}, {}, []
else:
    split_tables = {split: pd.DataFrame(read_riverscope_split(DATA_SOURCE, split)) for split in ("train", "valid", "test")}
    eligible_split_tables = {split: pd.DataFrame(filter_riverscope_rows(table.to_dict("records"), quality_exclusions)) for split, table in split_tables.items()}
    sample_rows = materialize_riverscope_sample(DATA_SOURCE, SAMPLE_ROOT, split=SPLIT, count=SAMPLE_COUNT, seed=SEED, excluded_image_paths=quality_exclusions)
    sample = pd.DataFrame(sample_rows)
    display(sample[["reach_id", "mid_lat", "mid_lon", "planetscope_id", "image_path", "mask_path"]])
    display(pd.DataFrame({"split": list(split_tables), "official_tiles": [len(table) for table in split_tables.values()], "eligible_tiles": [len(eligible_split_tables[split]) for split in split_tables]}))
    print(f"Loaded {len(quality_exclusions)} quality exclusions from {QUALITY_EXCLUSIONS_PATH.name}.")

## Non-zero value check

This automated check reads each PlanetScope tile and reports inputs whose image-band values are entirely finite zeros. It compares those results with the versioned exclusion manifest and does not modify the official archive or CSVs.

In [ ]:
if DATA_SOURCE is None:
    zero_value_tiles = pd.DataFrame()
    print("Load the dataset to run the non-zero value check.")
else:
    check_frames = [
        pd.DataFrame(find_riverscope_all_zero_images(DATA_SOURCE, split, progress_every=50)).assign(split=split)
        for split in ("train", "valid", "test")
    ]
    nonempty_frames = [frame for frame in check_frames if not frame.empty]
    zero_value_tiles = pd.concat(nonempty_frames, ignore_index=True) if nonempty_frames else pd.DataFrame()
    if zero_value_tiles.empty:
        print("No all-zero image tiles found.")
    else:
        zero_value_report = zero_value_tiles[["split", "planetscope_id", "normalized_planetscope_path", "total_values", "finite_values", "nonzero_values"]].copy()
        zero_value_report["in_exclusion_manifest"] = zero_value_report["normalized_planetscope_path"].isin(quality_exclusions)
        display(zero_value_report.sort_values(["split", "planetscope_id"]))
        print(f"All-zero image tiles: {len(zero_value_report)}")


## Technical inventory and spatial correspondence

RiverScope provides four PlanetScope bands in B, G, R, NIR order, 500×500-pixel tiles, and single-channel masks with values 0, 1, and 2. Radiometric values are not converted, and nodata is not replaced with land.

In [ ]:
def read_pair(row):
    with rasterio.open(row["image_path"]) as image_src:
        image = image_src.read()
        image_meta = {"bands": image_src.count, "shape": (image_src.height, image_src.width), "dtype": str(image.dtype), "crs": str(image_src.crs), "transform": image_src.transform, "resolution": image_src.res, "nodata": image_src.nodata, "bounds": tuple(image_src.bounds)}
    with rasterio.open(row["mask_path"]) as mask_src:
        mask = mask_src.read(1)
        mask_meta = {"bands": mask_src.count, "shape": (mask_src.height, mask_src.width), "dtype": str(mask.dtype), "crs": str(mask_src.crs), "transform": mask_src.transform, "resolution": mask_src.res, "nodata": mask_src.nodata, "bounds": tuple(mask_src.bounds)}
    return image, mask, image_meta, mask_meta

inventory = []
for row in sample_rows:
    image, mask, image_meta, mask_meta = read_pair(row)
    values, counts = np.unique(mask, return_counts=True)
    same_grid = image.shape[1:] == mask.shape and image_meta["crs"] == mask_meta["crs"] and image_meta["transform"] == mask_meta["transform"]
    inventory.append({"id": row["planetscope_id"], "shape": image.shape, "mask_shape": mask.shape, "image_dtype": image_meta["dtype"], "mask_dtype": mask_meta["dtype"], "crs": image_meta["crs"], "resolution": image_meta["resolution"], "image_nodata": image_meta["nodata"], "mask_nodata": mask_meta["nodata"], "aligned": same_grid, "label_values": dict(zip(values.astype(int), counts.astype(int)))})
inventory_df = pd.DataFrame(inventory)
display(inventory_df)
if inventory:
    observed = sorted(set().union(*(set(item["label_values"]) for item in inventory)))
    print("Observed values:", observed, "| expected: [0, 1, 2]")

## Gallery, shoreline zoom, and composites

The original mask is shown unchanged. The binary mask treats `1` and `2` as water. RGB and NIR/R/G composites are percentile-stretched only to make inspection easier. Review shifts, clouds, shadows, artifacts, and tile edges that do not represent physical shorelines.

In [ ]:
LABEL_CMAP = ListedColormap([(0.18, 0.19, 0.21), (0.08, 0.68, 0.91), (0.84, 0.31, 0.94)])
LABEL_NORM = BoundaryNorm(np.arange(-0.5, 3.5), LABEL_CMAP.N)

def composite(image, order):
    channels = image[list(order)].astype(np.float32)
    result = np.zeros((*channels.shape[1:], 3), dtype=np.float32)
    for output_channel, channel in enumerate(channels):
        finite = channel[np.isfinite(channel)]
        if finite.size:
            low, high = np.percentile(finite, [2, 98])
            result[..., output_channel] = np.clip((channel - low) / max(high - low, 1e-12), 0, 1)
    return result

def plot_gallery(rows, max_rows=4):
    rows = rows[:max_rows]
    if not rows:
        print("No samples available to display.")
        return
    fig, axes = plt.subplots(len(rows), 5, figsize=(18, 4 * len(rows)), squeeze=False, constrained_layout=True)
    for row_index, row in enumerate(rows):
        image, mask, *_ = read_pair(row)
        binary = np.isin(mask, [1, 2])
        rgb, false_color = composite(image, (2, 1, 0)), composite(image, (3, 2, 1))
        axes[row_index, 0].imshow(rgb); axes[row_index, 0].set_title(f"{row['planetscope_id']} · RGB")
        axes[row_index, 1].imshow(false_color); axes[row_index, 1].set_title("NIR / R / G")
        axes[row_index, 2].imshow(mask, cmap=LABEL_CMAP, norm=LABEL_NORM, interpolation="nearest"); axes[row_index, 2].set_title("Original 0/1/2 label")
        axes[row_index, 3].imshow(rgb); axes[row_index, 3].imshow(np.ma.masked_where(~binary, binary), cmap=ListedColormap(["cyan"]), alpha=0.38); axes[row_index, 3].set_title("Binary water")
        axes[row_index, 4].imshow(rgb)
        if binary.any() and (~binary).any(): axes[row_index, 4].contour(binary.astype(float), levels=[0.5], colors="yellow", linewidths=0.8)
        axes[row_index, 4].set_title("Boundary over RGB")
        for axis in axes[row_index]: axis.set_axis_off()
    plt.show()

plot_gallery(sample_rows)

if sample_rows:
    image, mask, *_ = read_pair(sample_rows[0])
    binary = np.isin(mask, [1, 2])
    boundary = find_boundary_pixels(binary)
    points = np.argwhere(boundary)
    y, x = points[len(points) // 2] if len(points) else (mask.shape[0] // 2, mask.shape[1] // 2)
    side = min(180, *mask.shape)
    x0, y0 = max(0, min(int(x) - side // 2, mask.shape[1] - side)), max(0, min(int(y) - side // 2, mask.shape[0] - side))
    region = np.s_[y0:y0 + side, x0:x0 + side]
    rgb = composite(image, (2, 1, 0))
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    axes[0].imshow(rgb[region]); axes[0].set_title("RGB zoom")
    axes[1].imshow(rgb[region]); axes[1].contour(binary[region].astype(float), levels=[0.5], colors="yellow"); axes[1].set_title("Zoom with boundary")
    axes[2].imshow(distance_squared_to_boundary(boundary)[region], cmap="magma"); axes[2].set_title("Squared distance to shoreline")
    for axis in axes: axis.set_axis_off()
    plt.show()

## Paginated dataset gallery

Use the controls to browse selected eligible tiles in a split. The gallery reads only the selected page and honors the versioned quality-exclusion manifest.

In [ ]:
from IPython.display import clear_output, display
import ipywidgets as widgets

PAGE_SIZE = 4
gallery_split = widgets.Dropdown(
    options=list(eligible_split_tables),
    value=SPLIT if SPLIT in eligible_split_tables else None,
    description="Split:",
)
gallery_page = widgets.BoundedIntText(value=1, min=1, max=1, description="Page:")
gallery_output = widgets.Output()

def page_count(split):
    return max(1, int(np.ceil(len(eligible_split_tables[split]) / PAGE_SIZE)))

def render_gallery_page(*_):
    split = gallery_split.value
    if split is None:
        with gallery_output:
            clear_output(wait=True)
            print("Load the dataset to browse its paginated gallery.")
        return
    total_pages = page_count(split)
    gallery_page.max = total_pages
    current_page = min(gallery_page.value, total_pages)
    if current_page != gallery_page.value:
        gallery_page.value = current_page
        return

    rows = eligible_split_tables[split].to_dict(orient="records")
    start = (current_page - 1) * PAGE_SIZE
    page_rows = rows[start : start + PAGE_SIZE]
    page_root = SAMPLE_ROOT / "pages" / split / f"page_{current_page:03d}"
    materialized_rows = materialize_riverscope_rows(
        DATA_SOURCE, page_root, page_rows, split=split
    )

    with gallery_output:
        clear_output(wait=True)
        stop = start + len(page_rows)
        print(f"{split}: page {current_page}/{total_pages} · tiles {start + 1}-{stop} of {len(rows)}")
        plot_gallery(materialized_rows, max_rows=PAGE_SIZE)

gallery_split.observe(render_gallery_page, names="value")
gallery_page.observe(render_gallery_page, names="value")
display(widgets.HBox([gallery_split, gallery_page]), gallery_output)
render_gallery_page()

## Distance and weight maps

`d²` depends only on the original mask. Derived maps use `w = 1 + alpha · exp(-d² / (2 · sigma²))`; changing sigma or alpha must not recompute the geometric transform.

In [ ]:
SIGMAS, ALPHAS = (1.5, 3.0, 6.0, 12.0), (1.0, 4.0, 10.0)
if sample_rows:
    d2 = distance_squared_to_boundary(find_boundary_pixels(np.isin(read_pair(sample_rows[0])[1], [1, 2])))
    fig, axes = plt.subplots(len(ALPHAS), len(SIGMAS), figsize=(16, 10), constrained_layout=True)
    for row_index, alpha in enumerate(ALPHAS):
        for column_index, sigma in enumerate(SIGMAS):
            axes[row_index, column_index].imshow(weights_from_distance_squared(d2, sigma, peak_weight=alpha), cmap="viridis")
            axes[row_index, column_index].set_title(f"sigma={sigma}, alpha={alpha}"); axes[row_index, column_index].set_axis_off()
    plt.show()
    print("The d² map was computed only once for this mask.")

## Class balance and splits

The audit looks for repeated IDs, shared `reach_id` values, repeated center coordinates, and unexpected labels. The test split is not used for hyperparameter selection.

In [ ]:
if split_tables:
    all_rows = pd.concat([table.assign(split=split) for split, table in split_tables.items()], ignore_index=True)
    display(all_rows.groupby("split").size().rename("tiles").to_frame())
    display(all_rows.groupby("reach_id")["split"].nunique().loc[lambda values: values > 1].to_frame("splits"))
    display(all_rows[all_rows.duplicated("planetscope_id", keep=False)][["split", "planetscope_id", "reach_id"]].sort_values("planetscope_id"))
    display(all_rows[all_rows.duplicated(["mid_lat", "mid_lon"], keep=False)][["split", "planetscope_id", "mid_lat", "mid_lon"]])
else:
    print("Load the dataset to audit the splits.")

## Record of favorable and problematic cases

Complete one row per tile after reviewing the plots. The score is a visual assessment of how closely the label follows the observable shoreline; it is not an accuracy metric.

In [ ]:
review_columns = ["planetscope_id", "reach_id", "split", "score_1_to_5", "notes"]
review = pd.DataFrame([{"planetscope_id": row["planetscope_id"], "reach_id": row["reach_id"], "split": SPLIT, "score_1_to_5": "", "notes": ""} for row in sample_rows], columns=review_columns)
display(review)
print("Save the completed table to data/processed/riverscope_quality_review.csv")

### Readiness criterion

Before training, images and labels must pass a visual review of alignment, nodata, artifacts, and shorelines. Problematic observations are retained alongside the tile identifier; labels are not silently corrected, and unreliable pixels are not treated as land.